<a href="https://colab.research.google.com/github/kunalpal425/Image-Classification-ML/blob/main/Final_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import tensorflow as tf

print("===== EcoSort AI: Fresh Setup =====")
print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

print("\nExisting model:")
model_path = "/content/ecosort_models/best_ecosort_model.keras"
print("Model exists:", os.path.exists(model_path))

print("\nPreviously downloaded dataset:")
zip_path = "/content/garbage-classification.zip"
print("ZIP exists:", os.path.exists(zip_path))

print("\nSetup check complete!")


===== EcoSort AI: Fresh Setup =====
TensorFlow version: 2.21.0
GPU available: False

Existing model:
Model exists: False

Previously downloaded dataset:
ZIP exists: False

Setup check complete!


In [ ]:

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

print("TensorFlow:", tf.__version__)
print("Available GPUs:", gpus)

if gpus:
    print("GPU is ready for training!")
else:
    print("GPU not detected. Check runtime settings.")


TensorFlow: 2.21.0
Available GPUs: []
GPU not detected. Check runtime settings.


In [ ]:

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.21.0
GPU devices: []


In [1]:

!pip -q install -U kaggle

from google.colab import userdata
import os

token = userdata.get("KAGGLE_API_TOKEN")

if not token:
    raise ValueError(
        "KAGGLE_API_TOKEN not found. "
        "Check Colab Secrets and notebook access."
    )

os.environ["KAGGLE_API_TOKEN"] = token

print("Kaggle token loaded successfully!")
print("Token value hidden for security.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.9 MB/s eta 0:00:00
Kaggle token loaded successfully!
Token value hidden for security.


In [2]:

!kaggle --version
!kaggle datasets list -s garbage-classification


Kaggle CLI 2.2.4
ref                                                      title                                                 size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------------  ----------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
asdasdasasdas/garbage-classification                     Garbage Classification                            85969666  2018-11-24 05:09:23.977000          78830        692  0.8125           
mostafaabla/garbage-classification                       Garbage Classification (12 classes)              250641573  2021-01-24 13:53:11.647000          44900        280  0.8125           
manonstr/tipe-webscraping                                Garbage classification dataset                  1197577469  2022-12-21 23:07:11.837000           1709         15  0.625            
sumn2u/garbage-classification-v2      

In [5]:

import os

dataset_slug = "mostafaabla/garbage-classification"
download_dir = "/content/ecosort_dataset"

os.makedirs(download_dir, exist_ok=True)

print("Downloading EcoSort AI dataset...")

!kaggle datasets download -d mostafaabla/garbage-classification -p /content/ecosort_dataset

print("\nDownload command finished.")
print("Files downloaded:")

for filename in os.listdir(download_dir):
    print(filename)


Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
100% 239M/239M [00:01<00:00, 153MB/s] 


Download command finished.
Files downloaded:
garbage-classification.zip


In [6]:

import os
import zipfile

download_dir = "/content/ecosort_dataset"
extract_dir = "/content/ecosort_dataset/extracted"

os.makedirs(extract_dir, exist_ok=True)

zip_files = [
    f for f in os.listdir(download_dir)
    if f.lower().endswith(".zip")
]

if not zip_files:
    raise FileNotFoundError(
        "No ZIP file found. Check whether the download succeeded."
    )

for filename in zip_files:
    zip_path = os.path.join(download_dir, filename)

    print("Extracting:", filename)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

print("\nExtraction completed!")
print("Dataset location:", extract_dir)


Extracting: garbage-classification.zip

Extraction completed!
Dataset location: /content/ecosort_dataset/extracted


In [7]:

import os
from collections import Counter

root_dir = "/content/ecosort_dataset/extracted"
extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

counts = Counter()

for root, dirs, files in os.walk(root_dir):
    images = [
        f for f in files
        if f.lower().endswith(extensions)
    ]

    if images:
        relative_path = os.path.relpath(root, root_dir)
        counts[relative_path] += len(images)

print("Dataset folders and image counts:\n")

for folder, count in sorted(counts.items()):
    print(f"{folder}: {count}")

print("\nTotal images:", sum(counts.values()))
print("Folders containing images:", len(counts))


Dataset folders and image counts:

garbage_classification/battery: 945
garbage_classification/biological: 985
garbage_classification/brown-glass: 607
garbage_classification/cardboard: 891
garbage_classification/clothes: 5325
garbage_classification/green-glass: 629
garbage_classification/metal: 769
garbage_classification/paper: 1050
garbage_classification/plastic: 865
garbage_classification/shoes: 1977
garbage_classification/trash: 697
garbage_classification/white-glass: 775

Total images: 15515
Folders containing images: 12


In [9]:

import os

root_dir = "/content/ecosort_dataset/extracted"
extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

total = 0

for root, dirs, files in os.walk(root_dir):
    images = [
        f for f in files
        if f.lower().endswith(extensions)
    ]

    if images:
        print(f"{os.path.relpath(root, root_dir)}: {len(images)} images")
        total += len(images)

print("\nTotal images:", total)


garbage_classification/paper: 1050 images
garbage_classification/brown-glass: 607 images
garbage_classification/cardboard: 891 images
garbage_classification/metal: 769 images
garbage_classification/white-glass: 775 images
garbage_classification/clothes: 5325 images
garbage_classification/shoes: 1977 images
garbage_classification/plastic: 865 images
garbage_classification/battery: 945 images
garbage_classification/biological: 985 images
garbage_classification/green-glass: 629 images
garbage_classification/trash: 697 images

Total images: 15515


In [10]:

import os

dataset_path = "/content/ecosort_dataset/extracted"
extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

if not os.path.exists(dataset_path):
    print("Dataset folder not found. Complete download and extraction first.")
else:
    total_images = 0
    image_folders = 0

    for root, dirs, files in os.walk(dataset_path):
        count = sum(
            f.lower().endswith(extensions)
            for f in files
        )

        if count:
            image_folders += 1
            total_images += count
            print(f"{os.path.relpath(root, dataset_path)}: {count} images")

    print("\nTotal images:", total_images)
    print("Folders containing images:", image_folders)


garbage_classification/paper: 1050 images
garbage_classification/brown-glass: 607 images
garbage_classification/cardboard: 891 images
garbage_classification/metal: 769 images
garbage_classification/white-glass: 775 images
garbage_classification/clothes: 5325 images
garbage_classification/shoes: 1977 images
garbage_classification/plastic: 865 images
garbage_classification/battery: 945 images
garbage_classification/biological: 985 images
garbage_classification/green-glass: 629 images
garbage_classification/trash: 697 images

Total images: 15515
Folders containing images: 12


In [11]:

import os
import tensorflow as tf

dataset_path = "/content/ecosort_dataset/extracted"

if not os.path.isdir(dataset_path):
    print("Dataset not found. Complete the download and extraction first.")
else:
    print("Checking dataset structure...")

    for root, dirs, files in os.walk(dataset_path):
        image_count = sum(
            f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".webp", ".bmp")
            )
            for f in files
        )

        if image_count:
            print(os.path.relpath(root, dataset_path), ":", image_count)

    print("\nDataset inspection finished.")


Checking dataset structure...
garbage_classification/paper : 1050
garbage_classification/brown-glass : 607
garbage_classification/cardboard : 891
garbage_classification/metal : 769
garbage_classification/white-glass : 775
garbage_classification/clothes : 5325
garbage_classification/shoes : 1977
garbage_classification/plastic : 865
garbage_classification/battery : 945
garbage_classification/biological : 985
garbage_classification/green-glass : 629
garbage_classification/trash : 697

Dataset inspection finished.


In [12]:

import os
import tensorflow as tf

dataset_path = "/content/ecosort_dataset/extracted"
image_size = (224, 224)
batch_size = 32

if not os.path.isdir(dataset_path):
    raise FileNotFoundError(
        "Dataset folder not found. Complete download and extraction first."
    )

train_dir = os.path.join(dataset_path, "train")
val_dir = os.path.join(dataset_path, "val")

if os.path.isdir(train_dir) and os.path.isdir(val_dir):
    train_path = train_dir
    val_path = val_dir
else:
    train_path = dataset_path
    val_path = None

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="int",
    validation_split=0.2 if val_path is None else None,
    subset="training" if val_path is None else None,
    seed=42,
)

if val_path is None:
    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_path,
        image_size=image_size,
        batch_size=batch_size,
        label_mode="int",
        validation_split=0.2,
        subset="validation",
        seed=42,
    )
else:
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_path,
        image_size=image_size,
        batch_size=batch_size,
        label_mode="int",
    )

print("\nClasses:", train_ds.class_names)


Found 15515 files belonging to 1 classes.
Using 12412 files for training.
Found 15515 files belonging to 1 classes.
Using 3103 files for validation.

Classes: ['garbage_classification']


In [14]:

import os
import zipfile
import subprocess
from google.colab import userdata

# Load Kaggle API token securely
token = userdata.get("KAGGLE_API_TOKEN")
if not token:
    raise ValueError("KAGGLE_API_TOKEN is missing from Colab Secrets.")

os.environ["KAGGLE_API_TOKEN"] = token

# Download dataset
download_dir = "/content/ecosort_download"
extract_dir = "/content/ecosort_data"
os.makedirs(download_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

dataset_slug = "mostafaabla/garbage-classification"

result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", dataset_slug,
    "-p", download_dir
], capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset download failed. Check the error above.")

# Extract ZIP
zip_files = [
    f for f in os.listdir(download_dir)
    if f.lower().endswith(".zip")
]
if not zip_files:
    raise FileNotFoundError("Download finished, but no ZIP file was found.")

for filename in zip_files:
    with zipfile.ZipFile(os.path.join(download_dir, filename)) as z:
        z.extractall(extract_dir)

print("Dataset extracted to:", extract_dir)


Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0


Dataset extracted to: /content/ecosort_data


In [15]:

import os

root_dir = "/content/ecosort_data"
extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
total = 0

for root, dirs, files in os.walk(root_dir):
    count = sum(
        f.lower().endswith(extensions)
        for f in files
    )
    if count:
        total += count
        print(f"{os.path.relpath(root, root_dir)}: {count} images")

print("\nTotal images:", total)

if total == 0:
    print("No images found. We need to inspect the ZIP structure.")
else:
    print("Images found. Dataset verification can continue.")


garbage_classification/paper: 1050 images
garbage_classification/brown-glass: 607 images
garbage_classification/cardboard: 891 images
garbage_classification/metal: 769 images
garbage_classification/white-glass: 775 images
garbage_classification/clothes: 5325 images
garbage_classification/shoes: 1977 images
garbage_classification/plastic: 865 images
garbage_classification/battery: 945 images
garbage_classification/biological: 985 images
garbage_classification/green-glass: 629 images
garbage_classification/trash: 697 images

Total images: 15515
Images found. Dataset verification can continue.


In [16]:

import os

possible_paths = [
    "/content/ecosort_data",
    "/content/ecosort_dataset/extracted",
    "/content/garbage_dataset"
]

extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

for base in possible_paths:
    print(f"\nChecking: {base}")

    if not os.path.isdir(base):
        print("Folder does not exist.")
        continue

    total = 0

    for root, dirs, files in os.walk(base):
        count = sum(
            f.lower().endswith(extensions)
            for f in files
        )

        if count:
            total += count
            print(f"  {os.path.relpath(root, base)}: {count} images")

    print("Total images:", total)



Checking: /content/ecosort_data
  garbage_classification/paper: 1050 images
  garbage_classification/brown-glass: 607 images
  garbage_classification/cardboard: 891 images
  garbage_classification/metal: 769 images
  garbage_classification/white-glass: 775 images
  garbage_classification/clothes: 5325 images
  garbage_classification/shoes: 1977 images
  garbage_classification/plastic: 865 images
  garbage_classification/battery: 945 images
  garbage_classification/biological: 985 images
  garbage_classification/green-glass: 629 images
  garbage_classification/trash: 697 images
Total images: 15515

Checking: /content/ecosort_dataset/extracted
  garbage_classification/paper: 1050 images
  garbage_classification/brown-glass: 607 images
  garbage_classification/cardboard: 891 images
  garbage_classification/metal: 769 images
  garbage_classification/white-glass: 775 images
  garbage_classification/clothes: 5325 images
  garbage_classification/shoes: 1977 images
  garbage_classification/pl

In [17]:

import os
import zipfile
import subprocess
from google.colab import userdata

dataset_slug = "mostafaabla/garbage-classification"
download_dir = "/content/ecosort_download"
extract_dir = "/content/ecosort_data"

os.makedirs(download_dir, exist_ok=True)
os.makedirs(extract_dir, exist_ok=True)

# Authenticate using Colab Secrets
token = userdata.get("KAGGLE_API_TOKEN")
if not token:
    raise ValueError("KAGGLE_API_TOKEN missing from Colab Secrets.")

env = os.environ.copy()
env["KAGGLE_API_TOKEN"] = token

# Download
result = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", dataset_slug, "-p", download_dir],
    env=env, capture_output=True, text=True
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(result.stderr)

# Extract
zip_files = [
    f for f in os.listdir(download_dir)
    if f.lower().endswith(".zip")
]

if not zip_files:
    raise FileNotFoundError("No ZIP file downloaded.")

for filename in zip_files:
    with zipfile.ZipFile(os.path.join(download_dir, filename)) as z:
        z.extractall(extract_dir)

# Verify
extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
total_images = 0
folders = {}

for root, dirs, files in os.walk(extract_dir):
    count = sum(f.lower().endswith(extensions) for f in files)

    if count:
        relative = os.path.relpath(root, extract_dir)
        folders[relative] = count
        total_images += count

print("\n===== DATASET REPORT =====")
print("Dataset:", dataset_slug)
print("Total images:", total_images)

for folder, count in sorted(folders.items()):
    print(f"{folder}: {count}")

if total_images:
    print("\nDataset download and verification successful!")
else:
    print("\nNo images found. Check the archive structure.")


Dataset URL: https://www.kaggle.com/datasets/mostafaabla/garbage-classification
License(s): ODbL-1.0
garbage-classification.zip: Skipping, found more recently modified local copy (use --force to force download)


===== DATASET REPORT =====
Dataset: mostafaabla/garbage-classification
Total images: 15515
garbage_classification/battery: 945
garbage_classification/biological: 985
garbage_classification/brown-glass: 607
garbage_classification/cardboard: 891
garbage_classification/clothes: 5325
garbage_classification/green-glass: 629
garbage_classification/metal: 769
garbage_classification/paper: 1050
garbage_classification/plastic: 865
garbage_classification/shoes: 1977
garbage_classification/trash: 697
garbage_classification/white-glass: 775

Dataset download and verification successful!


In [18]:

import os
import json
import tensorflow as tf

dataset_dir = "/content/ecosort_data/garbage_classification"

image_size = (224, 224)
batch_size = 32
seed = 42

if not os.path.isdir(dataset_dir):
    raise FileNotFoundError(
        f"Dataset folder not found: {dataset_dir}"
    )

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="int",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="int",
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("\n===== ECO SORT DATASET =====")
print("Number of classes:", num_classes)

for index, name in enumerate(class_names):
    print(f"{index}: {name}")

images, labels = next(iter(train_ds))
print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Pixel range:", float(tf.reduce_min(images)),
      "to", float(tf.reduce_max(images)))

# Save the exact label mapping for later mobile integration.
mapping_path = "/content/ecosort_class_names.json"

with open(mapping_path, "w") as f:
    json.dump(class_names, f, indent=2)

print("\nClass mapping saved to:", mapping_path)


Found 15515 files belonging to 12 classes.
Using 12412 files for training.
Found 15515 files belonging to 12 classes.
Using 3103 files for validation.

===== ECO SORT DATASET =====
Number of classes: 12
0: battery
1: biological
2: brown-glass
3: cardboard
4: clothes
5: green-glass
6: metal
7: paper
8: plastic
9: shoes
10: trash
11: white-glass

Batch image shape: (32, 224, 224, 3)
Batch label shape: (32,)
Pixel range: 0.0 to 255.0

Class mapping saved to: /content/ecosort_class_names.json


In [19]:

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Configuration
num_classes = len(class_names)
image_size = (224, 224)

# Load pretrained MobileNetV2 feature extractor
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers initially
base_model.trainable = False

# Build EcoSort AI classification model
inputs = keras.Input(shape=(224, 224, 3))

# MobileNetV2 expects pixels scaled to [-1, 1]
x = layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1
)(inputs)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.25)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax",
    name="waste_classification"
)(x)

model = keras.Model(inputs, outputs, name="EcoSort_AI")

# Configure training
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "EcoSort_AI"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ waste_classification (Dense)    │ (None, 12)             │        15,372 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,273,356 (8.67 MB)

 Trainable params: 15,372 (60.05 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [20]:

print("Model name:", model.name)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Number of classes:", num_classes)
print("Class names:", class_names)

assert model.output_shape[-1] == len(class_names)

print("\nEcoSort AI model is ready for initial training!")


Model name: EcoSort_AI
Input shape: (None, 224, 224, 3)
Output shape: (None, 12)
Number of classes: 12
Class names: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']

EcoSort AI model is ready for initial training!


In [21]:

import os
import tensorflow as tf

print("GPU devices:", tf.config.list_physical_devices("GPU"))

save_dir = "/content/ecosort_models"
os.makedirs(save_dir, exist_ok=True)

best_model_path = os.path.join(
    save_dir, "best_ecosort_model.keras"
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        best_model_path,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Training callbacks are ready.")
print("Best model will be saved to:", best_model_path)


GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Training callbacks are ready.
Best model will be saved to: /content/ecosort_models/best_ecosort_model.keras


In [22]:

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks
)


Epoch 1/10
388/388 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.7358 - loss: 0.8954
Epoch 1: val_accuracy improved from None to 0.93393, saving model to /content/ecosort_models/best_ecosort_model.keras

Epoch 1: finished saving model to /content/ecosort_models/best_ecosort_model.keras
388/388 ━━━━━━━━━━━━━━━━━━━━ 42s 77ms/step - accuracy: 0.8504 - loss: 0.5061 - val_accuracy: 0.9339 - val_loss: 0.2173 - learning_rate: 0.0010
Epoch 2/10
387/388 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9306 - loss: 0.2267
Epoch 2: val_accuracy improved from 0.93393 to 0.94264, saving model to /content/ecosort_models/best_ecosort_model.keras

Epoch 2: finished saving model to /content/ecosort_models/best_ecosort_model.keras
388/388 ━━━━━━━━━━━━━━━━━━━━ 20s 44ms/step - accuracy: 0.9345 - loss: 0.2122 - val_accuracy: 0.9426 - val_loss: 0.1728 - learning_rate: 0.0010
Epoch 3/10
387/388 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9448 - loss: 0.1755
Epoch 3: val_accuracy improved from 0.94264 to

In [23]:

# Evaluate EcoSort AI on validation data

val_loss, val_accuracy = model.evaluate(
    val_ds,
    verbose=1
)

print("\n===== EcoSort AI Evaluation =====")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")


97/97 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.9517 - loss: 0.1529

===== EcoSort AI Evaluation =====
Validation Loss: 0.1529
Validation Accuracy: 95.17%


In [24]:

import numpy as np
from sklearn.metrics import classification_report

y_true = np.concatenate([
    labels.numpy()
    for images, labels in val_ds
])

y_prob = model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))


97/97 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step
              precision    recall  f1-score   support

     battery       0.04      0.04      0.04       180
  biological       0.06      0.06      0.06       195
 brown-glass       0.03      0.03      0.03       111
   cardboard       0.06      0.07      0.07       176
     clothes       0.35      0.35      0.35      1065
 green-glass       0.04      0.04      0.04       122
       metal       0.05      0.06      0.05       145
       paper       0.06      0.06      0.06       218
     plastic       0.07      0.07      0.07       189
       shoes       0.13      0.13      0.13       407
       trash       0.04      0.04      0.04       143
 white-glass       0.05      0.05      0.05       152

    accuracy                           0.16      3103
   macro avg       0.08      0.08      0.08      3103
weighted avg       0.16      0.16      0.16      3103



In [25]:

import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report

# Load the best checkpoint saved during training
best_model = tf.keras.models.load_model(
    "/content/ecosort_models/best_ecosort_model.keras"
)

# Rebuild validation data WITHOUT shuffling
val_ds_fixed = tf.keras.utils.image_dataset_from_directory(
    "/content/ecosort_data/garbage_classification",
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32,
    label_mode="int",
    shuffle=False
)

# Confirm the model and dataset class order match
print("Dataset classes:", val_ds_fixed.class_names)
print("Model output classes:", best_model.output_shape[-1])

assert val_ds_fixed.class_names == class_names, \
    "Class order mismatch! Check class_names."
assert best_model.output_shape[-1] == len(class_names), \
    "Model output count does not match the number of classes."

# Evaluate the saved best model
loss, accuracy = best_model.evaluate(val_ds_fixed, verbose=1)
print(f"\nValidation accuracy: {accuracy * 100:.2f}%")
print(f"Validation loss: {loss:.4f}")

# Generate predictions in the same fixed order
y_true = np.concatenate([
    labels.numpy() for images, labels in val_ds_fixed
])
y_prob = best_model.predict(val_ds_fixed, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("\nCorrected classification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))


Found 15515 files belonging to 12 classes.
Using 3103 files for validation.
Dataset classes: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
Model output classes: 12
97/97 ━━━━━━━━━━━━━━━━━━━━ 9s 52ms/step - accuracy: 0.9726 - loss: 0.0970

Validation accuracy: 97.26%
Validation loss: 0.0970
97/97 ━━━━━━━━━━━━━━━━━━━━ 9s 63ms/step

Corrected classification report:
              precision    recall  f1-score   support

     battery       0.00      0.00      0.00         0
  biological       0.00      0.00      0.00         0
 brown-glass       0.00      0.00      0.00         0
   cardboard       0.00      0.00      0.00         0
     clothes       0.00      0.00      0.00         0
 green-glass       0.00      0.00      0.00         0
       metal       0.00      0.00      0.00         0
       paper       0.00      0.00      0.00         0
     plastic       0.00      0.00      0.00         

In [26]:

from sklearn.metrics import classification_report
import numpy as np

# Collect actual labels in the same order as predictions
y_true = np.concatenate([
    labels.numpy()
    for images, labels in val_ds_fixed
])

# Generate predictions using the best saved model
y_prob = best_model.predict(val_ds_fixed, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# Display metrics for all 12 waste classes
report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
)

print("===== EcoSort AI Corrected Classification Report =====")
print(report)


97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step
===== EcoSort AI Corrected Classification Report =====
              precision    recall  f1-score   support

     battery       0.00      0.00      0.00         0
  biological       0.00      0.00      0.00         0
 brown-glass       0.00      0.00      0.00         0
   cardboard       0.00      0.00      0.00         0
     clothes       0.00      0.00      0.00         0
 green-glass       0.00      0.00      0.00         0
       metal       0.00      0.00      0.00         0
       paper       0.00      0.00      0.00         0
     plastic       0.00      0.00      0.00         0
       shoes       1.00      0.99      1.00      1631
       trash       1.00      0.99      0.99       697
 white-glass       1.00      0.92      0.96       775

    accuracy                           0.97      3103
   macro avg       0.25      0.24      0.25      3103
weighted avg       1.00      0.97      0.99      3103



In [27]:

import os
import json
import tensorflow as tf

# Create a folder for final model files
final_dir = "/content/ecosort_final"
os.makedirs(final_dir, exist_ok=True)

# Save the best-performing model
final_model_path = os.path.join(
    final_dir, "ecosort_ai_final.keras"
)

best_model.save(final_model_path)

print("Model saved successfully!")
print("Location:", final_model_path)
print("Input shape:", best_model.input_shape)
print("Output shape:", best_model.output_shape)



Model saved successfully!
Location: /content/ecosort_final/ecosort_ai_final.keras
Input shape: (None, 224, 224, 3)
Output shape: (None, 12)


Save the class mapping

In [28]:

class_mapping = {
    str(index): name
    for index, name in enumerate(class_names)
}

mapping_path = os.path.join(
    final_dir, "ecosort_class_names.json"
)

with open(mapping_path, "w") as file:
    json.dump(class_mapping, file, indent=4)

print("Class mapping saved:")
print(json.dumps(class_mapping, indent=4))


Class mapping saved:
{
    "0": "battery",
    "1": "biological",
    "2": "brown-glass",
    "3": "cardboard",
    "4": "clothes",
    "5": "green-glass",
    "6": "metal",
    "7": "paper",
    "8": "plastic",
    "9": "shoes",
    "10": "trash",
    "11": "white-glass"
}


In [29]:

from google.colab import files

files.download("/content/ecosort_final/ecosort_ai_final.keras")
files.download("/content/ecosort_final/ecosort_class_names.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [30]:

import os
import tensorflow as tf

final_dir = "/content/ecosort_final"
os.makedirs(final_dir, exist_ok=True)

# Load the best trained model
best_model = tf.keras.models.load_model(
    "/content/ecosort_models/best_ecosort_model.keras"
)

# Convert the model to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

# Save the converted model
tflite_path = os.path.join(
    final_dir, "ecosort_ai_final.tflite"
)

with open(tflite_path, "wb") as file:
    file.write(tflite_model)

print("TFLite conversion successful!")
print("Model path:", tflite_path)
print(f"Model size: {os.path.getsize(tflite_path) / (1024 * 1024):.2f} MB")


Saved artifact at '/tmp/tmp8_u_4e1q'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 12), dtype=tf.float32, name=None)
Captures:
  138524497934096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497930256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497933712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497934288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497932368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497933136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497935056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497934480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497933520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497931408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138524497934

In [32]:

import tensorflow as tf

interpreter = tf.lite.Interpreter(
    model_path="/content/ecosort_final/ecosort_ai_final.tflite"
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("===== EcoSort AI TFLite Verification =====")
print("Input shape:", input_details[0]["shape"])
print("Input dtype:", input_details[0]["dtype"])
print("Output shape:", output_details[0]["shape"])
print("Output dtype:", output_details[0]["dtype"])

assert input_details[0]["shape"].tolist() == [1, 224, 224, 3]
assert output_details[0]["shape"][-1] == 12

print("Verification passed: model expects one 224x224 RGB image and returns 12 class scores.")


===== EcoSort AI TFLite Verification =====
Input shape: [  1 224 224   3]
Input dtype: <class 'numpy.float32'>
Output shape: [ 1 12]
Output dtype: <class 'numpy.float32'>
Verification passed: model expects one 224x224 RGB image and returns 12 class scores.


In [37]:

from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))

print("Selected image:", image_path)


Saving single-tightly-tied-black-plastic-260nw-2688570519.webp to single-tightly-tied-black-plastic-260nw-2688570519.webp
Selected image: single-tightly-tied-black-plastic-260nw-2688570519.webp


In [38]:

import numpy as np
from PIL import Image
import tensorflow as tf
import json

# Load class names in the exact training order
with open("/content/ecosort_final/ecosort_class_names.json", "r") as f:
    class_mapping = json.load(f)

class_names = [
    class_mapping[str(i)]
    for i in range(len(class_mapping))
]

# Load and prepare the image
image = Image.open(image_path).convert("RGB")
image = image.resize((224, 224))

image_array = np.asarray(image, dtype=np.float32)
image_array = np.expand_dims(image_array, axis=0)

# Load the TFLite model
interpreter = tf.lite.Interpreter(
    model_path="/content/ecosort_final/ecosort_ai_final.tflite"
)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Run inference
interpreter.set_tensor(input_details[0]["index"], image_array)
interpreter.invoke()

scores = interpreter.get_tensor(output_details[0]["index"])[0]

# Display the top 3 predictions
top_indices = np.argsort(scores)[::-1][:3]

print("===== EcoSort AI Predictions =====")
for index in top_indices:
    print(f"{class_names[index]}: {scores[index] * 100:.2f}%")

print("\nTop prediction:", class_names[top_indices[0]])


===== EcoSort AI Predictions =====
plastic: 51.46%
shoes: 23.47%
clothes: 20.98%

Top prediction: plastic
